# FL Initializer — Run Once Before Round 1
**FL-Pneumonia-Detection | Distributed Setup**

Creates the initial global model (pretrained EfficientNet-B0 with Opacus-compatible
GroupNorm layers) and uploads it to WandB Artifacts as `global-model:round-0`.

**Run this once.** After this, run hospital notebooks → aggregator for each round.

In [ ]:
!pip install opacus==1.4.0 wandb --quiet
print('✅ Packages ready')

In [ ]:
import os, io
import torch
import torch.nn as nn
from torchvision import models
from opacus.validators import ModuleValidator
import wandb
from kaggle_secrets import UserSecretsClient

WORK_DIR   = '/kaggle/working'
WANDB_PROJECT = 'fl-pneumonia-detection'
ARTIFACT_NAME = 'global-model'

wandb_key = UserSecretsClient().get_secret('WANDB_API_KEY')
wandb.login(key=wandb_key)
print('✅ WandB logged in')

In [ ]:
def fix_for_opacus(module):
    """Replace BatchNorm2d with GroupNorm recursively (Opacus compatibility).
    Avoids ModuleValidator.fix() which internally uses torch.load and breaks
    with PyTorch 2.6+ (weights_only=True default change)."""
    for name, child in list(module.named_children()):
        if isinstance(child, nn.BatchNorm2d):
            num_channels = child.num_features
            num_groups = min(32, num_channels)
            while num_channels % num_groups != 0:
                num_groups -= 1
            gn = nn.GroupNorm(num_groups, num_channels,
                              eps=child.eps, affine=child.affine)
            if child.affine:
                gn.weight.data.copy_(child.weight.data)
                gn.bias.data.copy_(child.bias.data)
            setattr(module, name, gn)
        else:
            fix_for_opacus(child)
    return module

def create_model():
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    num_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.2, inplace=True),
        nn.Linear(num_features, 2),
    )
    return fix_for_opacus(model)

print("Creating initial global model (pretrained EfficientNet-B0)...")
model = create_model()

errors = ModuleValidator.validate(model, strict=False)
status = "PASS" if not errors else str(errors)
print(f"Opacus validation: {status}")

# Smoke-test forward pass
_x = torch.randn(2, 3, 224, 224)
print(f"Forward pass: {list(_x.shape)} → {list(model(_x).shape)} ✅")
del _x

In [ ]:
# Save model checkpoint
checkpoint_path = f'{WORK_DIR}/global_model_round_0.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'round':            0,
    'description':      'Initial global model — pretrained EfficientNet-B0 (GroupNorm)',
    'classes':          ['NORMAL', 'PNEUMONIA'],
}, checkpoint_path)
print(f'✅ Checkpoint saved: {checkpoint_path}')

# Upload to WandB as artifact tagged round-0
run = wandb.init(
    project=WANDB_PROJECT,
    name='fl-initializer-round-0',
    job_type='initialize',
)

artifact = wandb.Artifact(
    name=ARTIFACT_NAME,
    type='model',
    description='Global FL model checkpoint',
    metadata={'round': 0, 'model': 'EfficientNet-B0-GroupNorm'},
)
artifact.add_file(checkpoint_path, name='global_model.pth')
run.log_artifact(artifact, aliases=['round-0', 'latest'])

wandb.finish()
print(f'✅ Uploaded to WandB as "{ARTIFACT_NAME}:round-0"')
print('\nNow run each hospital notebook (ROUND_NUM=1), then the aggregator.')